# Notes API Capstone -- Walkthrough

A guided tour of the Day 9 capstone project. Read each section, then jump to the **Demo** at the bottom to exercise the live API via `TestClient`.


## What we are building

A JWT-authenticated note-taking API that exercises everything from Days 1-8:

- REST design + versioned routes (`/v1`)
- FastAPI + Pydantic v2
- Middleware (timing) + dependency injection
- JWT auth (OAuth2 password flow), password hashing with bcrypt
- Rate limiting on auth endpoints via `slowapi`
- OpenAPI / Swagger UI customization
- SQLAlchemy 2.0 typed ORM (`Mapped`, `mapped_column`, `select`)
- Many-to-many relationships (notes <-> tags)


## Architecture

```
[Client] -> [FastAPI middleware] -> [Routers] -> [auth.py / db dep] -> [SQLAlchemy models] -> [SQLite app.db]
                                                  |
                                                  +-> [Pydantic schemas in/out]
```


## Endpoints

| Method | Path                       | Auth | Description                                  |
|--------|----------------------------|------|----------------------------------------------|
| GET    | `/`                        | No   | Health / landing                             |
| POST   | `/v1/users/register`       | No   | Create a new user (rate limited 5/min)       |
| POST   | `/v1/users/login`          | No   | Exchange creds for JWT (rate limited 5/min)  |
| GET    | `/v1/users/me`             | Yes  | Current user's profile                       |
| POST   | `/v1/notes/`               | Yes  | Create a note (optional `tag_names`)         |
| GET    | `/v1/notes/`               | Yes  | List my notes (`skip`, `limit`)              |
| GET    | `/v1/notes/{id}`           | Yes  | Get one of my notes                          |
| PATCH  | `/v1/notes/{id}`           | Yes  | Partial update; replaces tags if provided    |
| DELETE | `/v1/notes/{id}`           | Yes  | Delete one of my notes                       |
| GET    | `/v1/tags/`                | Yes  | List all tags                                |
| GET    | `/v1/tags/{id}/notes`      | Yes  | My notes carrying this tag                   |


## Install (run once)


In [ ]:
# Pinned versions match requirements.txt.
# Uncomment to install in this kernel:
# !pip install fastapi uvicorn 'pydantic[email]' 'python-jose[cryptography]' \
#     'passlib[bcrypt]' slowapi sqlalchemy python-multipart httpx


## `database.py` -- engine, Base, get_db

A single SQLite engine plus the FastAPI `get_db` dependency that opens a session per request and closes it on teardown.


In [ ]:
# Excerpt -- see database.py
from sqlalchemy import create_engine
from sqlalchemy.orm import DeclarativeBase, sessionmaker

DATABASE_URL = "sqlite:///./notes.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

class Base(DeclarativeBase):
    pass

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()


## `models.py` -- SQLAlchemy 2.0 typed ORM

Three tables (`users`, `notes`, `tags`) plus a `note_tags` association table for the many-to-many. Note the use of `Mapped`, `mapped_column`, and `relationship`.


In [ ]:
# Excerpt -- see models.py
class Note(Base):
    __tablename__ = "notes"
    id: Mapped[int] = mapped_column(primary_key=True, index=True)
    title: Mapped[str] = mapped_column(String(255))
    content: Mapped[str] = mapped_column(String)
    owner_id: Mapped[int] = mapped_column(ForeignKey("users.id", ondelete="CASCADE"))
    owner: Mapped["User"] = relationship(back_populates="notes")
    tags: Mapped[list["Tag"]] = relationship(secondary=note_tags, back_populates="notes")


## `schemas.py` -- Pydantic v2

Out schemas declare `ConfigDict(from_attributes=True)` so they can be built from ORM rows. `EmailStr` validates emails (falls back to `str` if `pydantic[email]` is not installed).


In [ ]:
# Excerpt -- see schemas.py
class NoteCreate(BaseModel):
    title: str
    content: str
    tag_names: list[str] = []

class NoteOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int
    title: str
    content: str
    tags: list[TagOut]
    created_at: datetime
    updated_at: datetime


## `auth.py` -- hashing, JWT, current-user dependency

`hash_password` / `verify_password` wrap bcrypt via passlib. `create_access_token` issues a signed JWT with an `exp` claim. `get_current_user` is a dependency that decodes the bearer token and returns the matching `User` row.


In [ ]:
# Excerpt -- see auth.py
SECRET_KEY = "change-me-in-production"   # read from env in real apps
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/v1/users/login")

def get_current_user(token=Depends(oauth2_scheme), db=Depends(get_db)) -> User:
    # decode token, fetch user, 401 on failure
    ...


## `routers/users.py`

`POST /register`, `POST /login`, `GET /me`. Register and login carry a `5/minute` slowapi rate limit.


In [ ]:
# Excerpt -- see routers/users.py
@router.post('/register', response_model=UserOut, status_code=201)
@limiter.limit('5/minute')
def register(request: Request, payload: UserCreate, db=Depends(get_db)):
    ...


## `routers/notes.py`

Full CRUD scoped to `current_user`. Every endpoint depends on `get_current_user`, so unauthenticated requests get 401. Tags are resolved by name via a small `_get_or_create_tags` helper.


In [ ]:
# Excerpt -- see routers/notes.py
def _get_or_create_tags(db, names):
    tags = []
    for name in names:
        tag = db.execute(select(Tag).where(Tag.name == name)).scalar_one_or_none()
        if tag is None:
            tag = Tag(name=name); db.add(tag); db.flush()
        tags.append(tag)
    return tags


## `routers/tags.py`

Read-only listing of tags plus a `/{id}/notes` endpoint that filters notes owned by the current user.


## `main.py`

Builds the `FastAPI` app, wires CORS + timing middleware + slowapi exception handler, calls `Base.metadata.create_all`, and mounts the three routers under `/v1`.


In [ ]:
# Excerpt -- see main.py
app.include_router(users.router, prefix='/v1')
app.include_router(notes.router, prefix='/v1')
app.include_router(tags.router, prefix='/v1')


---

## Live demo with `TestClient`

The cells below actually drive the running app in-process. They register a user, log in, create notes with tags, list and update them, and finally clean up.


In [ ]:
import os, sys
# Make sure the capstone folder is on sys.path so `from main import app` works
# regardless of where the notebook is launched from.
HERE = os.path.dirname(os.path.abspath('walkthrough.ipynb'))
if HERE not in sys.path:
    sys.path.insert(0, HERE)

# Use a throwaway database for the demo so we don't pollute notes.db.
os.environ.setdefault('PYTHONDONTWRITEBYTECODE', '1')
demo_db = os.path.join(HERE, 'notes_demo.db')
if os.path.exists(demo_db):
    os.remove(demo_db)

import database
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
database.DATABASE_URL = f'sqlite:///{demo_db}'
database.engine = create_engine(database.DATABASE_URL, connect_args={'check_same_thread': False})
database.SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=database.engine)

from fastapi.testclient import TestClient
import importlib, main
importlib.reload(main)  # re-run create_all against the fresh engine
client = TestClient(main.app)
client


### Register a user (`POST /v1/users/register`)


In [ ]:
r = client.post(
    '/v1/users/register',
    json={'email': 'demo@example.com', 'password': 'supersecret'},
)
print(r.status_code)
r.json()


### Log in (`POST /v1/users/login`)


In [ ]:
r = client.post(
    '/v1/users/login',
    data={'username': 'demo@example.com', 'password': 'supersecret'},
)
token = r.json()['access_token']
headers = {'Authorization': f'Bearer {token}'}
print(r.status_code)
token[:40] + '...'


### Who am I? (`GET /v1/users/me`)


In [ ]:
r = client.get('/v1/users/me', headers=headers)
print(r.status_code)
r.json()


### Create a note with tags (`POST /v1/notes/`)


In [ ]:
r = client.post(
    '/v1/notes/',
    json={
        'title': 'Buy groceries',
        'content': 'milk, eggs, coffee',
        'tag_names': ['errand', 'home'],
    },
    headers=headers,
)
note = r.json()
print(r.status_code)
note


### List my notes (`GET /v1/notes/`)


In [ ]:
r = client.get('/v1/notes/', headers=headers)
print(r.status_code)
r.json()


### Update a note (`PATCH /v1/notes/{id}`)


In [ ]:
note_id = note['id']
r = client.patch(
    f'/v1/notes/{note_id}',
    json={'title': 'Buy groceries (updated)', 'tag_names': ['errand']},
    headers=headers,
)
print(r.status_code)
r.json()


### List all tags (`GET /v1/tags/`)


In [ ]:
r = client.get('/v1/tags/', headers=headers)
print(r.status_code)
r.json()


### Delete the note (`DELETE /v1/notes/{id}`) + cleanup


In [ ]:
r = client.delete(f'/v1/notes/{note_id}', headers=headers)
print('delete ->', r.status_code)
print('list   ->', client.get('/v1/notes/', headers=headers).json())

# Remove the demo DB so re-running the notebook starts clean.
if os.path.exists(demo_db):
    os.remove(demo_db)
'cleaned up'


## Extension ideas

- Full-text search across `content` (FTS5 on SQLite or Postgres `tsvector`)
- File attachments per note
- Soft delete (`deleted_at` column + filter on every query)
- Refresh tokens + revocation list
- Per-user tag namespaces or shared workspaces
- Replace `create_all` with Alembic migrations
- WebSocket live updates when notes change


## Git workflow recap

- Branch per change: `feature/notes-crud`, `fix/jwt-expiry`, `docs/readme-endpoints`
- Conventional commits: `feat:`, `fix:`, `docs:`, `refactor:`, `test:`, `chore:`
- PR checklist:
  - [ ] `python -m py_compile` clean for every file
  - [ ] Manual smoke (this notebook) passes end-to-end
  - [ ] README endpoints table updated if routes changed
  - [ ] No `.db`, `.env`, or secrets in the diff


## Recap

You just exercised, in one project, every concept from Days 1-8:

- **Day 1** REST principles -- resource-oriented `/v1/notes`, `/v1/tags`
- **Day 2** FastAPI basics -- `APIRouter`, path/query params, response models
- **Day 3** Pydantic v2 -- `BaseModel`, `ConfigDict(from_attributes=True)`, `EmailStr`
- **Day 4** Middleware + DI -- timing middleware, `Depends(get_db)`, `Depends(get_current_user)`
- **Day 5** JWT auth -- bcrypt hashing, `OAuth2PasswordBearer`, signed tokens with `exp`
- **Day 6** Rate limit + versioning + docs -- `slowapi` 5/min, `/v1` prefix, OpenAPI metadata
- **Day 7** SQL + SQLAlchemy -- 2.0 typed models, `select`, session per request
- **Day 8** Relationships -- one-to-many (User->Notes) + many-to-many (Notes<->Tags)

From here, pick an extension idea above and ship it on a feature branch.
